<a href="https://colab.research.google.com/github/achinthajayaweera/medifind_app-SDGP/blob/OCR/ocr_prototype_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# Install packages (only needed once per session)
!pip install -q easyocr opencv-python-headless

# Imports
import easyocr
import cv2
import numpy as np
from google.colab import files
import os

#Basic Preprocessing
def preprocess_image(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return image_path

    # Denoise
    img = cv2.fastNlMeansDenoising(img, h=20)

    # Contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img = clahe.apply(img)

    # Adaptive threshold
    thresh = cv2.adaptiveThreshold(
        img, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        11, 2
    )

    # Very light dilation
    kernel = np.ones((1,1), np.uint8)
    thresh = cv2.dilate(thresh, kernel, iterations=1)

    proc_path = "processed_" + os.path.basename(image_path)
    cv2.imwrite(proc_path, thresh)
    print("Preprocessed image saved:", proc_path)
    return proc_path

#  Raw OCR extraction
def extract_raw_text(image_path):
    print("Initializing EasyOCR...")
    reader = easyocr.Reader(['en'], gpu=False, verbose=False)

    all_text = []

    # Original image
    print("\nOCR on original image...")
    result_orig = reader.readtext(image_path, detail=0, paragraph=True)
    all_text.extend(result_orig)

    # Preprocessed image
    proc_path = preprocess_image(image_path)
    print("\nOCR on preprocessed image...")
    result_proc = reader.readtext(proc_path, detail=0, paragraph=True)
    all_text.extend(result_proc)

    # Combine
    raw_text = ' '.join(all_text)
    raw_text = ' '.join(raw_text.split())  # just normalize spaces
    return raw_text

#  Main execution
print("\n" + "="*60)
print("    Upload your prescription image (jpg / png)")
print("="*60 + "\n")

uploaded = files.upload()

if not uploaded:
    print("No image was uploaded.")
else:
    image_filename = list(uploaded.keys())[0]
    print(f"\nProcessing image: {image_filename}\n")

    # Run OCR
    raw_output = extract_raw_text(image_filename)

    print("\n" + "="*70)
    print("RAW OCR RESULT (no cleaning, no correction)")
    print("="*70 + "\n")
    print(raw_output)
    print("\n" + "="*70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 17.9 MB/s eta 0:00:00

    Upload your prescription image (jpg / png)



Saving 114.jpg to 114.jpg

Processing image: 114.jpg

Initializing EasyOCR...



OCR on original image...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Preprocessed image saved: processed_114.jpg

OCR on preprocessed image...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



RAW OCR RESULT (no cleaning, no correction)

Date: Li4o_20to Aptus Medical Arts WE Hansen 123 Homestead PL Suite 2 Searchlight; NV 89046 Name: I~ Iyeeteive (MJF Address: MilkLmi_RL 1 Age: 51 Wt: 46kg L R Losnian 50 milliglam Ta6s , Dispense #30 Sig: T(ake on6 4 motizh daily in The 'momnin | koe Wood ueessevee conzeo€ Refill Gix times Signature 'WE_Hansen_Ip Generic Substitution 0K DEA #: Date: _ 6uiu1o.20to _ Aptus MedicalArts WVE, Hanscn 123 Hamnestcad pL Suitc 2 Searchlight NV 89046 Namez Ita11n Iypriteive Adldress; 4jolw Ru _ 1 Age: _ 54 Wt; 16kg [ R Losnan5o milliqiam Taea, Dispenae #30 Sig: Takoono ky month dailyi Thomorving ] fov lood peaoene contob Refill bit) times Signature WElanen._ID Generic Substitution 0K DEA #:

